# Exploratory Data Analysis
## Habit Drop Predictor — FitBit Dataset (33 Real Users)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

activity  = pd.read_csv('../data/raw/fitbit/dailyActivity_merged.csv')
sleep     = pd.read_csv('../data/raw/fitbit/sleepDay_merged.csv')
processed = pd.read_csv('../data/processed/real_user_processed.csv')
combined  = pd.read_csv('../data/processed/combined_dataset.csv')

activity['ActivityDate'] = pd.to_datetime(activity['ActivityDate'])
processed['date'] = pd.to_datetime(processed['date'])

print('Activity shape:', activity.shape)
print('Processed shape:', processed.shape)
print('Combined shape:', combined.shape)
print('Real users:', processed['user_id'].nunique())
activity.head()

## 1. Dataset Overview

In [ ]:
print('=== ACTIVITY DATASET INFO ===')
print(activity.dtypes)
print('\n=== MISSING VALUES ===')
print(activity.isnull().sum())
print('\n=== BASIC STATISTICS ===')
activity[['TotalSteps','VeryActiveMinutes','SedentaryMinutes','Calories']].describe()

## 2. Habit Completion Rates

In [ ]:
completion_rates = processed.groupby('habit')['completed'].mean() * 100
completion_rates = completion_rates.sort_values(ascending=False)

colors = ['#2ecc71' if v >= 70 else '#f39c12' if v >= 40 else '#e74c3c' for v in completion_rates.values]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(completion_rates.index, completion_rates.values, color=colors, edgecolor='white')
ax.axhline(y=70, color='green', linestyle='--', alpha=0.7, label='Good threshold (70%)')
ax.axhline(y=40, color='orange', linestyle='--', alpha=0.7, label='Warning threshold (40%)')
for bar, val in zip(bars, completion_rates.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_title('FitBit Habit Completion Rates (31 Days, 33 Real Users)', fontsize=14, fontweight='bold')
ax.set_xlabel('Habit')
ax.set_ylabel('Completion Rate (%)')
ax.set_ylim(0, 115)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/completion_rates.png', dpi=150)
plt.show()
print('Thresholds: Sleep>=420min | Workout>=20min | Steps>=8000 | Sedentary<=600min')

## 3. Weekday vs Weekend Completion

In [ ]:
processed['day_type'] = processed['is_weekend'].map({0:'Weekday',1:'Weekend'})
day_comparison = processed.groupby(['habit','day_type'])['completed'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(day_comparison.index))
width = 0.35
ax.bar(x-width/2, day_comparison['Weekday'], width, label='Weekday', color='#3498db', alpha=0.8)
ax.bar(x+width/2, day_comparison['Weekend'], width, label='Weekend', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(day_comparison.index, rotation=15)
ax.set_title('Weekday vs Weekend Completion Rate per Habit', fontsize=14, fontweight='bold')
ax.set_ylabel('Completion Rate (%)')
ax.set_ylim(0, 120)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/weekday_vs_weekend.png', dpi=150)
plt.show()

## 4. Habit Completion Trends

In [ ]:
habit_list = processed['habit'].unique()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, habit in enumerate(habit_list):
    hdf = processed[processed['habit']==habit].sort_values('date')
    daily_avg = hdf.groupby('date')['completed'].mean() * 100
    roll7 = daily_avg.rolling(7, min_periods=1).mean()
    axes[i].bar(daily_avg.index, daily_avg.values, alpha=0.3, color='steelblue', label='Daily Avg')
    axes[i].plot(daily_avg.index, roll7.values, color='navy', linewidth=2, label='7-day avg')
    axes[i].set_title(habit.replace('_',' '), fontweight='bold')
    axes[i].set_ylim(0, 120)
    axes[i].set_ylabel('%')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('FitBit Habit Completion Trends (Average Across 33 Users)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/habit_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
corr_cols = ['TotalSteps','TotalDistance','VeryActiveMinutes','FairlyActiveMinutes','SedentaryMinutes','Calories']
corr_cols = [c for c in corr_cols if c in activity.columns]
corr_matrix = activity[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0, mask=mask, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Between FitBit Activity Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150)
plt.show()

## 6. User Activity Distribution

In [ ]:
user_stats = activity.groupby('Id').agg(avg_steps=('TotalSteps','mean'), avg_active=('VeryActiveMinutes','mean'), avg_sedentary=('SedentaryMinutes','mean')).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].hist(user_stats['avg_steps'], bins=15, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=8000, color='red', linestyle='--', label='8000 steps threshold')
axes[0].set_title('Avg Daily Steps per User', fontweight='bold')
axes[0].legend()

axes[1].hist(user_stats['avg_active'], bins=15, color='green', edgecolor='white', alpha=0.8)
axes[1].axvline(x=20, color='red', linestyle='--', label='20 min threshold')
axes[1].set_title('Avg Very Active Minutes per User', fontweight='bold')
axes[1].legend()

axes[2].hist(user_stats['avg_sedentary'], bins=15, color='orange', edgecolor='white', alpha=0.8)
axes[2].axvline(x=600, color='red', linestyle='--', label='600 min threshold')
axes[2].set_title('Avg Sedentary Minutes per User', fontweight='bold')
axes[2].legend()

plt.suptitle('FitBit User Activity Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/user_activity_distributions.png', dpi=150)
plt.show()

## 7. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = combined['will_drop'].value_counts()
axes[0].pie(counts.values, labels=['Stable','Will Drop'], colors=['#2ecc71','#e74c3c'], autopct='%1.1f%%', startangle=90, explode=(0,0.05))
axes[0].set_title('Target Variable Distribution', fontweight='bold')

drop_by_habit = combined[combined['will_drop']==1]['habit'].value_counts()
axes[1].bar(drop_by_habit.index, drop_by_habit.values, color='#e74c3c', alpha=0.8, edgecolor='white')
axes[1].set_title('Drop Events by Habit Type', fontweight='bold')
axes[1].set_xlabel('Habit')
axes[1].set_ylabel('Number of Drop Events')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150)
plt.show()
print(f'Drop events: {counts[1]} ({counts[1]/len(combined)*100:.1f}%)')
print(f'Stable     : {counts[0]} ({counts[0]/len(combined)*100:.1f}%)')

## 8. Feature Distributions

In [ ]:
feature_cols = ['roll7_rate','streak','days_since_miss','habit_age']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    stable   = combined[combined['will_drop']==0][col]
    dropping = combined[combined['will_drop']==1][col]
    axes[i].hist(stable,   bins=30, alpha=0.6, color='green', label='Stable',    density=True)
    axes[i].hist(dropping, bins=30, alpha=0.6, color='red',   label='Will Drop', density=True)
    axes[i].set_title(col.replace('_',' ').title(), fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend()

plt.suptitle('Feature Distributions: Stable vs Will Drop', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150)
plt.show()

## 9. Real vs Synthetic Comparison

In [ ]:
combined['user_type'] = combined['user_id'].apply(lambda x: 'Synthetic' if str(x).startswith('synthetic') else 'Real FitBit')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_map = {'Real FitBit':'#2E75B6','Synthetic':'#00A896'}

for j, (utype, grp) in enumerate(combined.groupby('user_type')):
    cr = grp.groupby('habit')['completed'].mean() * 100
    axes[0].bar(np.arange(len(cr))+(j*0.35), cr.values, width=0.35, label=utype, color=colors_map[utype], alpha=0.8)

axes[0].set_xticks(np.arange(len(cr))+0.175)
axes[0].set_xticklabels(cr.index, rotation=15)
axes[0].set_title('Completion Rates: Real vs Synthetic', fontweight='bold')
axes[0].set_ylabel('Completion Rate (%)')
axes[0].legend()

drop_rates = combined.groupby('user_type')['will_drop'].mean() * 100
axes[1].bar(drop_rates.index, drop_rates.values, color=[colors_map[k] for k in drop_rates.index], alpha=0.8)
axes[1].set_title('Drop Rate: Real vs Synthetic', fontweight='bold')
axes[1].set_ylabel('Drop Rate (%)')
for i,(idx,val) in enumerate(drop_rates.items()):
    axes[1].text(i, val+0.05, f'{val:.2f}%', ha='center', fontweight='bold')

plt.suptitle('Real FitBit vs Statistically-Derived Synthetic Users', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/real_vs_synthetic.png', dpi=150)
plt.show()
print('Synthetic data mirrors real FitBit behavioral patterns.')

## 10. Key Findings Summary

In [ ]:
print('='*60)
print('  KEY FINDINGS — FITBIT DATASET')
print('='*60)
print(f'1. Real FitBit users  : {processed["user_id"].nunique()}')
print(f'2. Days tracked       : 31 (Mar-May 2016)')
print(f'3. Total combined rows: {len(combined):,}')
print(f'4. Drop event rate    : {combined["will_drop"].mean()*100:.1f}%')
cr = processed.groupby('habit')['completed'].mean()*100
print(f'5. Highest completion : {cr.idxmax()} ({cr.max():.1f}%)')
print(f'6. Lowest completion  : {cr.idxmin()} ({cr.min():.1f}%)')
wd = processed.groupby('is_weekend')['completed'].mean()*100
print(f'7. Weekday: {wd[0]:.1f}% | Weekend: {wd[1]:.1f}%')
print(f'8. Synthetic users    : 50 (from real FitBit distributions)')
print(f'9. Test set           : 7 real FitBit users (unseen)')
print('\nAll figures saved to reports/figures/')
print('='*60)